# Multi-Signal Short Squeeze Scanner

Three-stage pipeline combining:
1. **SEC Failure-to-Deliver (FTD) data** — persistent delivery failures as a proxy for hard-to-borrow / heavily shorted stocks
2. **Volume/Price technicals** — volume spikes, Bollinger Band squeeze, price momentum
3. **SEC filing sentiment** — LLM-powered analysis of filings for squeeze catalysts, ownership dynamics, and short seller vulnerability

### Scoring (0-100)
| Signal Group | Max Points | Key Components |
|---|---|---|
| FTD Signals | 30 | FTD ratio, trend, persistence |
| Volume/Price | 35 | Volume spike, BB squeeze, momentum, market cap |
| SEC Sentiment | 35 | Catalyst strength, float pressure, short vulnerability |

The final `buy_signal` is determined by LLM consolidation (not a raw score threshold).

## Setup

In [ ]:
import sys, os
import pandas as pd
sys.path.append("../")

from wallstreet_quant.short_squeeze import ShortSqueezeScanner

FMP_KEY = os.getenv('financial_modeling_prep_api_key')
assert FMP_KEY, "Set the financial_modeling_prep_api_key environment variable"

## Load Stock Universe

In [ ]:
import json

with open("company_tickers.json") as f:
    sec_data = json.load(f)

tickers = [v['ticker'] for v in sec_data.values()]
print(f"Universe: {len(tickers)} SEC-listed tickers")

## Initialize Scanner

In [ ]:
scanner = ShortSqueezeScanner(
    fmp_api_key=FMP_KEY,
    batch_delay=0.25,       # seconds between FMP API calls
    model="gpt-5.2-pro"     # model for SEC sentiment analysis
)

## Run Full Scan

**Stage 1** (all tickers): Compute short interest + FTD + volume/price signals. This is cheap — only yfinance/FMP API calls + SEC FTD downloads.

**Stage 2** (filtered only): Tickers passing all three gates (`si_score_min`, `ftd_score_min`, `preliminary_score_min`) get SEC filing sentiment analysis via 3 specialized LLM calls each.

**Stage 3** (filtered only): LLM consolidation produces a final assessment + buy/no-buy signal.

Adjust the three gate thresholds to control how many tickers reach the expensive LLM stage. Lower = more candidates but higher cost.

In [ ]:
results = scanner.scan(
    tickers,
    si_score_min=18,           # min short interest score (0-25) — ensures meaningful short exposure
    ftd_score_min=19,          # min FTD score (0-20) — confirms delivery failures
    preliminary_score_min=50,  # min combined si+ftd+vp (0-80) — overall signal strength
    run_sec_sentiment=False     # set False to skip LLM analysis (Stage 1 only)
)

## View Buy Candidates

In [ ]:
candidates = results[results['buy_signal'] == True].sort_values('composite_score', ascending=False)
print(f"Buy candidates: {len(candidates)}")
display(candidates[['ticker', 'composite_score', 'ftd_score', 'vp_score', 'sec_score',
                     'assessment', 'buy_signal']])

## View All Screened Tickers

In [ ]:
display(results[['ticker', 'price', 'composite_score',
                  'short_pct_float', 'days_to_cover', 'short_interest_change', 'si_score',
                  'ftd_score', 'ftd_trend',
                  'vp_score', 'bb_squeeze', 'volume_dryup', 'volume_spike_ratio',
                  'catalyst_strength', 'float_pressure', 'vulnerability_level',
                  'sec_score', 'assessment', 'buy_signal']].head(30))
print(f"Total screened: {len(results)}")

## Save to Excel

In [ ]:
filepath = scanner.save_results(results)
print(f"Saved to: {filepath}")

## Debug: Single-Stock Analysis

Examine all signal components for a specific ticker.

In [ ]:
# Pick a ticker to inspect
test_ticker = "GME"

# FTD signals
ftd_df = scanner._load_recent_ftd_data()
ftd_signals = scanner.compute_ftd_signals(test_ticker, ftd_df)
print("=== FTD Signals ===")
for k, v in ftd_signals.items():
    print(f"  {k}: {v}")

# Volume/Price signals
vp_signals = scanner.compute_volume_price_signals(test_ticker)
print("\n=== Volume/Price Signals ===")
for k, v in vp_signals.items():
    print(f"  {k}: {v}")

# SEC sentiment
sec_signals = scanner.compute_sec_sentiment(test_ticker)
print("\n=== SEC Sentiment ===")
for k, v in sec_signals.items():
    print(f"  {k}: {v}")

# Composite
composite = ftd_signals['ftd_score'] + vp_signals['vp_score'] + sec_signals['sec_score']
print(f"\nComposite Score: {composite:.1f}/100")